# V84 — автономное воспроизведение прогноза

Этот ноутбук запускается отдельно от проекта: снимок всех 21 входных CSV и манифест встроены в него. Внешние файлы проекта не требуются. Нужны `numpy` и `pandas`.

V84 калибрует прогноз V61 по сохранённым прогнозам и оценкам лидерборда; это не модель, обученная только на исторических регистрациях. `0.39140` — сообщённый Public score исходного результата, ноутбук его не вычисляет. Часть входных score имеет пометки `historical_markdown_unverified` или `projected_not_measured` в манифесте.

Запускайте ячейки по порядку. В конце создаются `forecast_2024_v84_standalone.csv` и JSON с метаданными в текущей папке ноутбука.


In [ ]:
# Если numpy или pandas отсутствуют, раскомментируйте и выполните эту строку один раз:
# %pip install numpy pandas

import base64
import hashlib
import json
import tempfile
import zipfile
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd


In [ ]:
# Встроенный архив: manifest + базовый прогноз + 20 прогнозов-направлений.
BUNDLE_B64 = (
'UEsDBBQAAAAIANEqVV0HOhHHuAcAAAEbAAANAAAAbWFuaWZlc3QuanNvbrVZ227jyBF9368g/Gw7fb/MTwRYBHkJFkR1d7XFjEQK'
'JOWsdzD/nkPZHnt3ycmOYsGwYbGb1VWHdTmH+vJT09wceN4N5eZTc7NnKjymgcbSVuaSKH9uM+27NNLcDf3N7bI/D33tHrD/Cz7h'
'88gPpz2N3W/Pez414l7b2+c1mqbTgUvLj7Q/0Yz/Cj1N2OP0vbh9vb/yyH3mdsrDyGcDRhn9sjzT+MBz+zjsYQmLUhrxemveDxNs'
'TjSfxhfD/zqvYE0JZe6Eu1Pq5vb318KdMn++puXN+dIv+Pv1HGiiifddz2+hHmneLUB1/fE0T3+rcDfTNLeLjfbRyXYYgeXD0NO+'
'rae+dP3DfZ4eXw67mXakrFsMVCWrE9V60pmzq4ZY2ZS0TIZD0blUHcjjaaTorZUeGxIFksoVb7z2N++d7N7i/vIa1fc9tbGdzb15'
'59vvvDMl56pSISJrYs3RFcHC55Biyk76kCmEJKIuFKTXKdsobajOScvKZP/O5vsn6vXvr7efu/6cd6eJx3bkI9Dj8nbzMHYPXX+G'
'6y+4f6A573iCnYdumsen9ls883ji866vtz8CkhMtUh3oztzPLUwyHaYtyPCIOEknZVHKhpyt1KGkYBILx8k6E1QlYWO22hQrSpFG'
'U9bKsIwuWbEOmdPugyD7C8F8PIC6nZimoV8sHXkLuqqR58GS8oJVyJaCDsoFT8gYRvZJ5iDJmRx8qLFSqRXlIbBSFWvOq9BpJOZH'
'Qfe9MD4cNG/bjD7cb9amrUIWisplr9hVZ5zxwYQsAzCMAVbQIWWVSheftHZZpxDJooaBb3CraHmtwwehte7/x8MU2+mUjNiCyYXK'
'3mQSvoRcbJJh6VVOB1bZmaI5size51xUIWGKybqIAEhTETX4sgoTOp78KJhW/d+GqdJ+ugSnILcQSl5lVVSVxkth2VhHpKtysgAx'
'WxNXF01yigFSQOciSsaWUoW3TlNYb/LCmo/qWH/0/ArY6C1sAmYdmyXiEqyipBQZjXrCnDMm5qKFdSyspJpKUCSRYNoraVwmTG1r'
'1ltS/Kga+6Pj/391PQbTHsduWNxY5gTuy/BlM3cSUhf1ZKOK4DAC1YbWw0oYG1xh0j4Gy7ZUIocOXVXOWcbsXMyxkNZr+Oho5VZ1'
'Hcfh37w41PbD3B7QjU/jOlI/v3e/OY782A2nqflnMM146j81Xd/NHe0bGe/OoTfTE0bi4bah/XFHz9S1KUMG1+wXC8/ss3lmn41U'
'Qst7pe6bv59PhKE8FG72oLZjs5y8sN2mm6dm+E/fDKcZR9zfXJCbLxTHbz0BbY0KqSYnKAHmCtKoSqEsZY2YDcIEwbrW6FmX4p2Q'
'taDEZQxGiuq8+FGKtkNiAWWIgfZA4+eC8NpT/8hjV7sf4WzPAV0CSGh3tK//i4fZAnaMyuSFkHlD4K4xFMBifVLZo7k5wRSrTsah'
'xSVB0Rdnqwtkc3Lrfd8Yr66Gy0pcl8DjXyx802vb1Yta9A4NDBmjDCmVbHaRUaml5MQBANYMMgG8VHAiWptMAPPSDrBpndcpRHTO'
'Xw2jreAuAcq1h9N+7l4MYghPR4L25LQFl/fFZrXwg6gDWJeAYotZRQyBZUxYJ5KuIsvqVfXaeCtNoiK9DOCuNqw2OxMwKq8G1/dD'
'vAQ0207d4bjnX3X5DlLko4PuAcmySjNIPDnOnlxRNltonyxkDrZmDbFovKckScZaYhRViOTsBjcNVwNqJaxL0DHtNNS5nXcjT7th'
'/z2IFDCwUilUYE05hqXUoO9NBh9Fq3bSIJtS0JIMyJeAZjxPWWGDjShFvQGRclfDaCu4S4DS7e6UwHU4tXSah344YDhvQSWiWCg6'
'kkWJ6DjGGsl7J1PxVRd09MpGaa1yDTqlyCkRNHcBbFFh9NX1uovyakhtR3cJVuqdEVCemTZfPUirirFZklfW5mos+QLiWiuXiKtR'
'cwL3jNA3OkYPJLmA+zPuYZFLWsfJ2etRgfXQLgFJtA8j9QXnnw/fgigGlJzyjIEGwYO5J1mBzEPTeCNYMC5YnVyOHgQvQ9xodHVm'
'MCsPNlvsRgu/XtGtxXUBPia045BO+LxwUwY93aw2zK/MS/tB2bni2GKmeYEkAZ/XIiJ1ksB0Y3YQO5mridGapJJnb63yG6RAXK3a'
'tkK7BCV/1uK+5R5nbb+nUkJqKSNajgGntN7pKEWuWShUnHIFFMGGJH0QYORZa1FDZaeRaQxOsJ5EMeqr0aa1uC6Bx7YHyKYD/bqY'
's5uvXDCjZE0gGtlCmgSKGFwEShlLRIXhRxplwY4gBpXwTi3VCPWYLRKKstrMoKv1odXAngHC31/Or9dfVTl08QRO9fZVQO32vCbN'
'zfoLfzDFmoJk0B5OtuCXRY2FVRIJAgT8EjgJb1NxAfkkU2WsYcE6qyK/gnNzPKV9l9vxMO1fZbM0r6cNpzHz1guFt1AK+POZiwjl'
'7qS4U/Lt64R38r8b+kVun7f+Y8dNHYffuG/OrwnOgvpZQHdTU4D5I3QvdhyaGVvfCehvEvyb4oYOv21guKndfJbYQ7PrSoHpQt3+'
'CWI68X66b37mO2ztu/7hbHPPD5SfmimP3XFu6IE6+HleyadxxGFN6eA9UuIJ5/N0PgIxj0M5ZT5vfEWgWR4eZPnXn/4LUEsDBBQA'
'AAAIANEqVV1I7eUcjgEAAOEEAAAvAAAAaW5wdXRzL2ZvcmVjYXN0XzIwMjRfdjYxX29ydGhvZ29uYWxfZnVuZGluZy5jc3ZVlEtO'
'ZDEMReesxWXFv8RZDoJi2lI3+1ffgMDOMEcvvrZzqt5fP5/08efv8+313+eLDvXHmA8dZMGSBYRs84wCSoNHHe0+OokkxywSJBqs'
'VmSSpHHLWKQ6eTWSpBE8VpFNYsaziI2WvB5DKNbkHUWUIiaLFDEQ5VxFnGJiYCsSFJo8s8hEM4NtF1l3dCLaeLegTYG1jSorg0Ki'
'RwsaNuHKESVHLzWAGKlIrQ7Er2AJ3Jjs7cokN+XtRRa5ovtZJPGko9YNsvHqg6OC4IEEoqsyROjR0MCm9qXABAvn2Yr4V9nZigQZ'
'2tNq70sG4Vm7gw05WNoneSfDhOGc+ktggu62SBOCTv5TIY8bALsBhXGcUcAoeew690XnkcKcrc5QwqvpPEYcyVcBGCxs7QraXuwV'
'ASOSfdZZrkjYgKF2VTw2GFsDfoDUVNDhzoQN4n3uI8N1I+/MfaZoizsa5AXkTCHVNTwQ5ahNQgP8lKsHvTcJA64jOoRmWWDhdTlb'
'QB7g7YvzT9BbwPNvXvUW9r3I/1BLAwQUAAAACADRKlVdAXaD1JIBAADeBAAAIQAAAGlucHV0cy9mb3JlY2FzdF8yMDI0X3Y1OV90'
'NC40LmNzdlWTS24dQQhF51kLDxXfopZjOc7UkuP9K2ApDTXs87q5fM77/fb9AX8+vz7e3/5+/+LF+lr+4gXiKKcBgS5kbcCwcPWj'
'3I8KxAuPNbEkimdUcKAwJGqygXmje5MANsLx0QESQn6ArBG8X4vA9kLbTRgsS1gDSWAYgyiYO0Y0MTDhJzeBZyMH1yD7Dg6wyJhR'
'9YDpRjoPoQVWO5Am9JNjXZYY1KU3l0SASa46ekWTgebMMYo4aLXfA9EGZcHwJgFyCPd454CoI/fuUgMyRulm0oMZnRaIHzzdW4og'
'FuijiCaRcYBUQURRqUmpQBjjow07cPem0oQrOD1YaWjPkyKkGMQNCNjx/J8vyo0ELg04fUMdQCC6RJQYHRplhUg3FT9WjEGipCjQ'
'z+nvamuipCB/dhGlxJ49El2JaQMf5PG7VMvPdqNcSBDdc9qQkWOolIEU93gjXUAZNePOPDVDz1QSZI/egGqGPQDnfxt1AMmpVify'
'vccU4HrMBqn9jjo+rXmoPH6Csfg6P7fNUdc/7XLU8SviH1BLAwQUAAAACADRKlVd4EkLw5ABAADfBAAALwAAAGlucHV0cy9mb3Jl'
'Y2FzdF8yMDI0X3Y2MF9jb25zaXN0ZW50X3N0cmVhbXMuY3N2VZPLTgYxCEb3Pgs/KXf6OMbL1kR9/0jHKMxyTlo+Cmden7/f4P3j'
'8+3l+ev7iRfrY/mDF4ijcwMCXZjZgGHh6k+5fyoQLzRtYkCyUKOJA6WjNAjgc4SaJLAT+iizq4xjNyZrBMdjEVhsTG7CYF6tDCJg'
'5pjRROvWwt3AwITRrIkDa6Bkk7gnJ1gGaoNdMYzRN2iBsaFIk2pXCYWaMGg46jgjwGujjTN6SyYDzZpbN0sOWmXZmwQo87yUIHvN'
'F9IGMcPV/ZYFZIrRZ0qDmVwSiDuuHlx5IPXo0X6pIOpIgxhI7dl6/pcKgjbqBNQaV7+5VLgllwe0ZnKJUEWl5y8EHPg3gjxqcP7v'
'OI8YrOjRQCBwjwtzzHmcEGuP8lKiVNQGcRJpROSJsBGxa5ftcx4lcvRIdIssGaTssAblgiFLA70SuqmSgWU2VS5QyT6ulArIo2be'
'M/d51b+ReVmw56tKAsr2JI8GJD36PBbEnCzfJ1kC3D6rQ0YfAXHATMzadv8Bea1fkRrU9ksH70nK7yR/AFBLAwQUAAAACADRKlVd'
'jm73vpABAADkBAAAKQAAAGlucHV0cy9mb3JlY2FzdF8yMDI0X3Y2M19zZWFzb25fc2hhcGUuY3N2XZNNbp4hDIT3OYs/y3+AOU7U'
'pttKbe6vDIr02mTpR8AMw/D7/fOD/vz99/Hr/f/nm4nFS+bLhDw5rYBSCOcuYCQsNfo9BqkapxYZpLZ4ZZFJuoJ3E1lkZpdsksXi'
'EUU2TlaW+RCXJr1eojTm5L2KGI1Q9kYcRDhGkQAx9igyaMgswyCTzMe1a93SSWMZr7Zg0zhB6kNUaOjirDUKw+osVsQoYrKWGXXS'
'rRztnLikdVDMzdIWTAqXnoMuCm1hgiQ5DHczm9yRVZlBFTRgr9agC10aTfApbHUsyuC+WduWAPEeL/rgiDfL3unDNN4FFoLpF0Ab'
'LmFUQTYvfwiqAN1dIbgSXuy5X552WFzA0Lm6XZ5u4NVnzT3oPLVAPjMLzHNirALrgDbnUbC2A76Rjj0AnVgsUbNekqiD7T47esjh'
'BeIoZEmiDSa8doGTLUt5OF2otuRpwiW5f3g+LcCnqlhQAgCtE1AD/PhRrtCCZCsPdgeJAlwjHCpnE1h4Xc5mIc+KFtt5f++XwPNj'
'y6gc/DvIL1BLAwQUAAAACADRKlVdNG3uS44BAADyBAAAIgAAAGlucHV0cy9mb3JlY2FzdF8yMDI0X3Y3NV9jbGVhbi5jc3Zdk0tu'
'HUEIRedZC0b8C5ZjJc40UuL9K5SVCPoN66iKC/TpH++fH/Dz1++P7+9/Pr8Jib1RvAmBKmoMYFDBkwMECInmrC9ngzLUBRyYBGOV'
'CGAlVB9yQJTx8JAECUNZrwoqMQco7eTzRgxeK6iJgHuh1RAFD8ewIdak8MgQBxdDOkOi2zP0GHJeshPCFHPdKPDj6FOFqQmjzyvu'
'jrVmyCYCdgpz+mMFIUJexJ7Z3P2SYUw2B5gGEg85YByoq0qCUWCu/grUHlO2Ddz9pQ7hZ3bboKFYM0H7oK2QTVIboZIj1blKqPTO'
'V91Woue2Vbk7NswF8iW6gNmRh+iXvGt3yiAH6/+IeRW5IAYI9KeWdaMFwaA5P5adV47uXGpA3Aq6KpwLfGXkVwYPKJD2YECb0Rvz'
'OfMztK2QwFgXWgpFnox2ogHpAL8ZazC+G37U6D5RVki+hHaXtHd1bfDRLK8M9wPYAAEWtHVD+9+fL5/XhUdIm/A8x31Q02VrULvr'
'tqAvbFA3c83RFjCjznb13zb/AlBLAwQUAAAACADRKlVdf8AJDBoCAADQBQAAIgAAAGlucHV0cy9mb3JlY2FzdF8yMDI0X3Y3OV9z'
'dWI0MC5jc3ZVlEmuGDcMRPc+Cz/BmdRxDA/bAInvj1Tb8Ke6dypIVLH41N+//vpBP//598e3r//9+mJi8SH1YUIhPKdSakWlVE6d'
'nFzRSFh26e9lkJazjLjOqkk6wqWnzFct7BVWwXdWbTJrrkm9nAxZJrt75lXgkGpyKxxuAZfLUX+IUlVzj830qkblyQdf6qpOeZwz'
'2o+tGpTZrClW196kOMZeWjOrFpnr477LV+23paHKYlU/elY9lHEQd87Za1RgdNjc+jMPqJhMBmvMia2gRmHFFdqVqzrCT66uc4nx'
'cqRJKcFhcTvSokBIM5Z23d3k5yCOUN2QdABQ8YjMFagecocjbev4VAEbTnN3SG9d0HZbAmeB4Z7BdNcScPMyhGTtew2o88Rh9/ar'
'YGLvcEnUrNGHuomHRZOdG6hTR6DWk5fReVsCb1N8zGUdATfgGhOSW86VQIGgbfu7cx4SMZ0slbxEI8POMc/VnAqsK1hc7R7YPPQ9'
'+JbUqRUBX3IEsLjEfm4NhGC6It6TACqRvqygvWQzd7FPEfwpfgypWmtQ9WUG3KHbzMrYe4EdGktgdxULMoQsJXZtzMe16+m8bqjf'
'Xkbs83HNb/IY/yLXNa3ztnLIhsdNNjrgBieNsWmsqKSHDQ9o6wM5dTCAh75O7BkGcD2+lu09DID2WhYdPmiye7X+E6O/Ss8jlmZd'
'uT2QOXf6uY4DMvwZQiS3A/8zhP8BUEsDBBQAAAAIANEqVV3EHA+6FAIAANMFAAAcAAAAaW5wdXRzL2ZvcmVjYXN0XzIwMjRfdjgx'
'LmNzdlWUS85VQQiE566Fn/BsYDnGx9RE3X+so4n0md3zpS9dBUV//fz7G33/8fPbl8+/fn8ysfiQ82FC3nzkWNdCpTgc1u250EhY'
'9tPfn0F6lLVlXJcmaQa7hJYvPTh7uNxO29IiC+VJ/OM626DNI9keS4fUgiVVZWW7XJLqQ5RONVsfXL/U6Ehzt9mcpU55BgWt86JB'
'GcN+cI8tTUoZrnHRXHrIUjhbz1y31VtSU45z+4TO0qFMxxC00v9TFTpekG8TF1VIKq7suISqUbhzKlytUHXCTzZLl6tCvCRpUoyx'
'YBZTSw8FJEmPVC8tCiv2jlu+NjlMzTO3S9KQh3Mc7bN1ETfNgVCMLpbqSxKS5lOQ1FrbSgTO0/iMyOzlyJ17cHjWuc4maHKL6yX0'
'yV0LD+bvFy1SCTapuqwidy9JCFwjSzp+TReBc4HF6C3nSgaDbvXfdj9RtGY96PIyIzOGQyR7odPhFCubZffA+okfYuEa2rnwSR+y'
'0hbXDfUXWh7ThU06HDmWvRD2sKaVu6b9BFAFolsv1aovMQieKxc60NcZf5wFHpW5qgUZEup6bFk+WqZaJxZiSo7HYUoviORxSLRd'
'FfstZZ4eW5bqukXeTBh7ZLMNRdy0OcOwcAsNTwpEw8qyZxpRcfXU3sNA0F6f52lZZE6u9r8BQ4+s6tKAMShLJRZh4TzWsa2n9yRC'
'hicoT1w98n9T+ANQSwMEFAAAAAgA0SpVXQSlxKgYAgAA1gUAABwAAABpbnB1dHMvZm9yZWNhc3RfMjAyNF92ODMuY3N2VZRLjhUx'
'DEXnrCVtxX97OS1opkjQ+xc3rxFODeson+v4uH68f36sn79+f3x///P5TbbY24432cs2VXikDuRlSp3MIQNlbdrzqc9PW6xOzNZx'
'HeQvWhksF40lO6nKpH1oLtEmjW63obXEk7LTq4Y2zt0Uvr0moe4rUr5tXh5F7HtXDJXlVuSWYjZUX2s121qH2vJW2hzqNdSxFhdt'
'buWhKCoRlKM6h+YzUq1IIQmpvg7sFSLkrFVDeS8vpdiWeyLxKaopLSSnKJZljYsCl08k1sWNSIoXlaH2iMSohk81qXJtDRigJHj2'
'vGguYxwYLXJFqoV3ozJlv2gvNSE8Bnf/p9CNPUhUZftQfkSCadpO6E/atUiXwoTKFpvK4d25pjmzr7UOysRdfQWFdxCaUALA0Fx+'
'FM2o6Ru0eySCb5Dm9KLnbvimaGYKMg3kJYUCC4v/wToqKuI4RNKBAmMgV+7igbqSUHdf6+6G1dEPIwWB1GNgvM4yjKwPxEQFWbmZ'
'DDwDRYGB0h7YS5ii1Hu2Q8Agd5aa3cyPLPAOY+joiEwJ0E6UEv8UvzbaEoOKATzQDzTMgtbA83egM1x9QZhHvT1rHpXrmeWrBjQ8'
'prCXb9RSmIyBDA/Qt/QJDecYI9hbdw7EAG0SdI2vI5/tgGqPTzimxPizXS8CxQD3dolpGRRjo114k3mmIxpkxB/renR4hu3evK+W'
'6Vcn/gJQSwMEFAAAAAgA0SpVXWU5onAaAgAA0wUAACIAAABpbnB1dHMvdjg0X3ByaW9yX3JlY29uc3RydWN0ZWQuY3N2VZRLjlUx'
'DETnrCVt+Rd/ltOCZooEvX9ReYh27jClxCm7TvLj/fNj/fz1++P7+5/Pb8rqbxxvysuZxHenjyhrM6mzmIyoi4lnac+lL9lJsplz'
'xL0kgjxQ/KoeUDdtdi4bNZeakzL3fWkt3UIVKNGj9hIpEhG3UY0vR/nGssKVssRaR9UV6pRhFTmqrV1OYsoSo/ra2ync2Ebcyzso'
'dkRdZWOpJ204+poj1Hw6qhXs5JqsV8F+3eJeqvKlCq9ABpIpOY4EwcC9l+x9qbrcglIrrwK2pJWEs+Pa6g9HgmYK42W36lED9Yos'
'tOw6mssaKlK61QI/mFyb5lWhl2Eczap7hgTWZCv5NpMxCthuS8DMxWlH89UNaENgpBWyZ76AzkCvWdoVJqgzh9EsRD8qEmK0ioh6'
'jII6gf1y9DRNgbqHJeB26HDdF0nAzeTVTF6iLFMKibb/NuuQqE3W8G4jKninfYCrEW0Ba1bu6/QdWR3+TjZ4JSwjorkg7vIvWOrQ'
'p0kRWfdOPCg87Wzl61o0CM7Nwuc4CGzKtrY9mjy8ADw8WcPn4TmincaEd9l0C+4gdmVexfZSAFrKoSPGsSLqu8bfQY96b8lrYz2t'
'gDjG08JPM1YAHG5gLem5FrxJI7fsa1agTpQ6EjCPaAvIJd7sdfqZBlh7LOGe8bnAxvR+GBN8dy18Wauz009g14WvHOTwPKKdHIKZ'
'a1zYvxz+AlBLAwQUAAAACADRKlVdw/6xWpEBAADjBAAAIgAAAGlucHV0cy9mb3JlY2FzdF8yMDI0X3Y1OV90NC40Ny5jc3ZVk0tu'
'niEMReddi2PhF5jlRGk6rdRm/+rlj6rLN+QI/Dz8fP/6lF+//3x+vP/9+uHD823MNx8SUz0JTHKoXcBl6OAxnscU86GLoABSZ5FM'
'sU7tTbLEfWlcpMXL1I1ki4Wp8U6MK/N6Gya1kDlJXApBcpMESGkWSUrNqXm9Kikk2otkopjWNUnWM3VL9dBwki2VpZvEhpSHNsOa'
'fScKEpecodEkIW7BrkHykdoKT/Zdm03JcB1GsiSRuq6wLbFNF+dgWyKxeBYDE6wQh6+gwp0aIgRSB4PAhcCgitVCh6jQeYUtiUjl'
'vF86oJbrypLV96RgwyMxVBgYLnuGCpjk1XKY+NT+X3sfOwAqCBzOsfg+brTadeEedB8tAo1sAliR3HkfKQ64IrTY5n77WGGT6+0j'
'xdLNmGaPnPDBNz9jv3RwnTznOa8LlOD3xSLAdJPz75cMGteLfqbcpwljV0cD1Mg5wQI0sRgRHphrGkHIfLx4ThIGPI6oEP/6er6w'
'Xi32gPUDTNZ8BHBtFon977uk+J7jP1BLAwQUAAAACADRKlVd1ntjLYoBAADkBAAAKQAAAGlucHV0cy9mb3JlY2FzdF8yMDI0X3Y1'
'OF9oYWxmX3N0cmVhbXMuY3N2VZNLbgUgCEXnXQuPCIjgcpp+pk3a7j+9tknBoSfgRT2+Pn+/0fvH59vL89f3kw6dj7EeOmhOXl5A'
'aDpHA0qDRy3tXk4SMw4t4iQTJVlkkezFEkWCFDW7QJKuwdliNzZWHlVjoyXHYwj5Up5eRMld2aKIkYfzyiKTPJPDiji6jLcUWaRh'
'PGaRuKOTFm5Nd5FNS5S9omUgerNWtPwNvBtRmhsDV5AYqRhLq5lXtGBaxV2uIgsPtng0EjQtL5JkG69UYJPNZKsSiCDLedeZYEJP'
'hgdzKGvdN1QwHKh2hQw2g70uFzKYOUud8MiAB2jPBhlCr5K8g2GCXrvCBFsc9WImZG20PHKYcKwCCsFK6zxqBKfUut9zHivMr3pI'
'kZxaAA4Hr1kADjtny9ykTaM8TgRrZYpcmdABQ88oYGfo3jAPkAb8RHgDGBN71tyQwXnWUJJ35u+MYv/geJD9FNAAwLKA4oPzai1G'
'IvXt8njQMyDBtYQBGKn1x9lwtP48Fa1jn4I2EwTAL2kd9neTP1BLAwQUAAAACADRKlVd6+03jpcBAADgBAAALgAAAGlucHV0cy9m'
'b3JlY2FzdF8yMDI0X3Y1N19zdHJlYW1fY2FsaWJyYXRlZC5jc3ZVlEluJDEMBO/zFjbBXdRzDC9XA7b/D1MzmCZ1VHSVkkpF9dvL'
'zzt8fH69v758//wREntQPITABcMaMMTClQ0ECKmXei8NWBbabuLAbLi9SQDvxIwmCyQCacQmyEokbrKBVdGbKI3o9aCaVOoBbyLg'
'TijSRCFIkAaxIhuTmzh4Ju7VJEBNkHeTdUcnLDHUscmGUB+PcBW7HHc/wgweVUMDAUufw7GC1HWoNrErmGtWI+SxSYAJdXFFFhgp'
'ZvfCCboNZWy7QRfNE5YIQoR7EL6iywPThdnTlgqagmZNDDSKNHBQT4zu9q8Mhj1uuVDrHQ3yzj0e7PZpHQ9sVq1ch0H9P3keM3Rc'
'aB4x1NryPF6wtOV5tOjQPE6UAOPnMsKRtcE64ClnHiWUMQbY5TiaPEEZsXD3DMxX5LGh+u+1ng2fH00eFSry2UMeGeoDGTtEfUN9'
'p3lMWOhjh7wT90nwPuVxwDF65FKgzpBdZEnAOYYsBVhnsXL3WPd/LevyF9JIXMBxjVD/A4w0EvcB3qeu269XuIvWfz3+AlBLAwQU'
'AAAACADRKlVdS33DyCYBAAAaBAAANAAAAGlucHV0cy9mb3JlY2FzdF8yMDI0X3Y1Nl9tdWx0aXN0cmVhbV9zdWJzcGFjZV9lYi5j'
'c3ZVkztuAzEMRPuchQbEj0TqOEbstAES3x+hiuyMyweJHIpv93F/PeXr++f5ef99fdiwuI11syERIBXfIJMBcIYQGxM4RbcDl6gv'
'YPYpZZSYUciWzIt8XCl5GypLJ9Bk5gK6zDBgSGzCKZEbuMQ8gckxJXNR5ZY16kIdPR+h9hCYSa1jMJO6aNHloBidMjdG0NWNMKBm'
'j099q7VgJt3i0y9saWoBVIppab4Q09rcFdjikhq1uELMEVd4zRGndLk4ZrdjYHtbeLerxH9dHY3X11DHotOZCx1hXXUEWoLa3wTl'
'21m90eabxx5AqX+bo6oWx1W9JQP1JAE6XxIohaC4/+Z7bYxJxRRkvXVQf0JIM15J2yJYb1XnHwOVKNZ6VOFtbUoxiZ+V/AFQSwME'
'FAAAAAgA0SpVXRe4zZQlAQAAGQQAACkAAABpbnB1dHMvZm9yZWNhc3RfMjAyNF92NTVfc2ltcGxleDNkX2ViLmNzdlWTS27EMAxD'
'9z0LA1iSP/JxBu10W6Cd+6NMgYbK8kGWRPslH4/XE59f38/3x8/rzZv3o83DGyJFhtgiRxNEhQ5bQzjgXnCyWoYuVk2Y8BnCDTO/'
'MNq1Zh3NMK41RMcYXRismrCz6sKB0VM44bscXnVNYrZS25imQdaQoaox01YIY6amiBZw02TrZY0x0ZjCieFbuNB3OZzopvi2EUtV'
'WrMeQitraC2mOuktutbQXLjynuZCmf7M6a40t8qWrFs29rqI1rreJPgV/bflKfF6vjwdVgosgR4rT32eItobIn5QpS1vtCvR3RRY'
'mU9v9VzU+bTmyk9t3kXzRgulLet85hDRVyWDbZHDNIS6TLu9PgldFZi3rkVvooTp2ucPJqKp0hfnk/wCUEsDBBQAAAAIANEqVV3s'
'T1ohIgEAABkEAAAuAAAAaW5wdXRzL2ZvcmVjYXN0XzIwMjRfdjU0X3NvZnRfdGhyZXNob2xkX2ViLmNzdlWSS04EMQxE95ylWoo/'
'cZzjjGDYIsHcX7iR6HIvn+K4HL98PF5PfH59P98fP683HerHiEMHbJMEliTFIFgHhywlTqgaMSDZiledtuKExiRuSDu1ccWsYwhm'
'JFELJ9EwlxEdc27ixPSGARtCXD0mEdcIhRshfqEMrGSq1EwZRIVvjihWb2WMeIuRmigYI4Epi7jg2U4TPlrqhk2+tazJZOfS1kDL'
'KG+WN/OGDhOu5c8cU09zwfHLXFtwiWspG8lnl7WWYfWL/ilPiWakmk5IhkXgsvLU1+sC2grrQyUpb7R7ZbkLgrT+5U03qbS1Qody'
'/tKmToobLbTC7P1rDr2ofCnPSpcwu3wJm5Qu4ba0r6RcNYjbrVXeSHmj3clGv2fnSn4BUEsDBBQAAAAIANEqVV34sTXaJwEAABkE'
'AAAwAAAAaW5wdXRzL2ZvcmVjYXN0XzIwMjRfdjUzX2h1YmVyX2ViX2F1dG9ub21vdXMuY3N2VZNJTgNBEATvvCUtdS29PccCc0UC'
'/19UIzGZPoZq7YqZj/vzgc+v78f7/ef55s3z1sbNG7KRDDFJDgmFQsI0scOyEwdsBXHC3YkLPiS6q9W4MNo1Zt6aoUcnOvpVWRiF'
'ScxX7Og5iQPhizh1zMLog7gxGiutoU8n1k5doo6crLWAJze2lDFWG3UjjqqURhOZXNAWYghuhPFxZc3kraWNY8paBq9U3iIkmojG'
'JcqcN4455lKiE1OmLJ2yIUcpa3LrqK/o/yTrSIxF8vJACkgij7WOvusDW3/2jDRfqC4ldRu+Lyp3bGIm/cub9DjaJDG1Y2lzLlnW'
'PEgTnbC0/9a844s3KF3OzPJlbFK6TOr0JOVKYLxUTVxfxzqijK85PxipTElmnJP8AlBLAwQUAAAACADRKlVd7/RLHCIBAAAZBAAA'
'LAAAAGlucHV0cy9mb3JlY2FzdF8yMDI0X3Y1Ml9hdXRvbm9tb3VzX21ldGEuY3N2VZMxbsQwDAT7e8saEEmJkp5zSC5tgOT+j9Ap'
'vOtyIHJJa+TP5/uFr++f18fz9/3w5v1oeXhDN5IhkuRohFDosOnEAY8gZp1u4oSb4IKn9G5YZ3K0a8w8mmF0JzrGCmJgTCnuGCnF'
'o3o7MREhxVPHLGRK7kYag6xhGoOsdsokOrosYQHvi9hljNVGMsYSfU3iRB8ydZUJCdoIY1RZM1mitMlZbTSYW97CBTt8D+KALY45'
'zYUUT0wuXOJkytajstbZFobL2jolxiZ5rUMKTAIva536tCtvXfNGdVPSt+HronKXBJP88na92PWvTbq6JpY2l5Csp06aGISl+bUH'
'j05f/JrS5UZymFRGGSDplZQrgbx1zRut+v9IW6lMXY95naba4w9QSwMEFAAAAAgA0SpVXaqnmQ0nAQAAGQQAACoAAABpbnB1dHMv'
'Zm9yZWNhc3RfMjAyNF92NTBfZ3JhbmRfdW5pZmllZC5jc3ZVk0luw0AMBO95SxsYLrM9x4ida4DE/0dah0zTx4K4tFjS4/564uv7'
'5/l5/319ePO8tXHzhmwiQ0yRozyKCgmLEHbiEA7YTuGEexcueC9LNsz9YLSzZt6aoW8TOnobwkBPFyb67MLO3oIDvkrxrGsWxnkb'
'4sZo+6A19KFOMwxbQmay8jTgJZNlWWNM1LXGBnIpoE1imbsQ54bEjejKRGs2NJnaCjgiNYjeomkNzUUoIM15lrk0l7owzU1loLiy'
'ZWOqkNZShcGv6N/auiRWcvgWBYZAx1qXvtpFe1PED6q0rbeJm54P0Z3azMp8eqt1UefTWplPbZWYpIsmCqw6nzn80OVL26jLlJi+'
'TPOpy0JUT0JXBcZb14QpCEWZ3ub6wVRJU5ai6yR/UEsDBBQAAAAIANEqVV2yeIPpIQEAABkEAAAtAAAAaW5wdXRzL2ZvcmVjYXN0'
'XzIwMjRfdjQ4X3JvYnVzdF9jb25zZW5zdXMuY3N2VZNJbgMxDATveUsbEBdR1HOM2LkGSPx/hDrEzTkWxL1mHvfXE1/fP8/P++/r'
'Q4f6bcRNB2yTBBYkxSBYB4eoEidEWl5AMokLqpOY0Gi5u3IZbOPdZt2GYK4gKnwb0TBNiF7Bkzgxs+UGbChx9TaJkBa6C/kq49JG'
'aqbNNqKYylyxWm4TvbWRmsgbBjwbrsIkJny01w1bHL+sSbBraWNoWTN1otXirFvmTBtO6OS8x9xsbRaiVcreZaNdt6w5q1h9Rf8H'
'yyPRBkkvZAgCj5VHn25SQFtgfVCTlJfIjfd6edwxTaTVL2+t4tG2SN7rl7YeGZe3hQbZ69ccvMHxxalKl3Di8iWsX7rESf0k5apB'
'XLJWeSMlhNucH4yRZarVt3OSP1BLAwQUAAAACADRKlVd/YUpISIBAAAZBAAAKQAAAGlucHV0cy9mb3JlY2FzdF8yMDI0X3Y0N19z'
'dWI0N19lbmdpbmUuY3N2VZJBbsRACATveQuWBpgB5jmrZHONlOz/lXakuPGxhN20KX88Xk/5/Pp+vj9+Xm82bB4jDhvim6TiQTIZ'
'BO8wRbWIC7iIIVqTmGLWdpRYJHGLDqKPa00eQ2XFJpqsEUSXNZ04ZVXDBSxiiI8WlX1NSdgkbiCnOmR5EtEpWULRybhGXSwZpbOt'
'UTRyI4bMatOUmW1NwUvL3eLJh2FNgyWgrc1M3HkHeHPlh8PcDRdksO9pLlpUSiqp+pYt7diwNhnq+Iv+M+uU6INkN8K5CDxWnfr6'
'c4GzkvBDBQn/0yZtuXTU6Y4j1ZYPby3jTxtpii3SuhGa8NsgrY2q56OHXnT6YivoUtaCL2UT6FLmWz8JXDWI21spyiIQ1We7E0y1'
'fD9P8gtQSwMEFAAAAAgA0SpVXUPTj3MmAQAAGQQAACoAAABpbnB1dHMvZm9yZWNhc3RfMjAyNF92NDVfbWluaW1heF9zdWI1MC5j'
'c3ZVk0tOBDEMBfec5Y0U27HjHAfBsEWCub9wFvRzL0vxv7o/319PfH3/PD/ef19vOnQ+Rjx0wDZJYEFSDIJ1mBBVokOk5QUkW9EF'
'VScmNFrurty80MbVZj2GwNckKuY2osFNiLOCnejwbBiwocTV2yRCFnEjxr5Qxq2N1EybM4nCNYhWyyVxtjZSE1mrG5ir4cJMDiFZ'
'Xtrrhi0OUdYkuFxpY5uyZtrerBbnRGXOtKFDnW2OOeeVylzw3iWuddlIrl3WJqtYfUX/s+aRaIOkNzIEgcfKo083KaAtsD4oJ+Ut'
'cuNaL487pom0+uWtVTzaFmn2+qWtR8btbaFB9vo1B29wfHGq0iWcuHwJ65cumaR+knLVIG5Zq7yREsJtzg/GyDLV6ts5yR9QSwEC'
'FAMUAAAACADRKlVdBzoRx7gHAAABGwAADQAAAAAAAAAAAAAApIEAAAAAbWFuaWZlc3QuanNvblBLAQIUAxQAAAAIANEqVV1I7eUc'
'jgEAAOEEAAAvAAAAAAAAAAAAAACkgeMHAABpbnB1dHMvZm9yZWNhc3RfMjAyNF92NjFfb3J0aG9nb25hbF9mdW5kaW5nLmNzdlBL'
'AQIUAxQAAAAIANEqVV0BdoPUkgEAAN4EAAAhAAAAAAAAAAAAAACkgb4JAABpbnB1dHMvZm9yZWNhc3RfMjAyNF92NTlfdDQuNC5j'
'c3ZQSwECFAMUAAAACADRKlVd4EkLw5ABAADfBAAALwAAAAAAAAAAAAAApIGPCwAAaW5wdXRzL2ZvcmVjYXN0XzIwMjRfdjYwX2Nv'
'bnNpc3RlbnRfc3RyZWFtcy5jc3ZQSwECFAMUAAAACADRKlVdjm73vpABAADkBAAAKQAAAAAAAAAAAAAApIFsDQAAaW5wdXRzL2Zv'
'cmVjYXN0XzIwMjRfdjYzX3NlYXNvbl9zaGFwZS5jc3ZQSwECFAMUAAAACADRKlVdNG3uS44BAADyBAAAIgAAAAAAAAAAAAAApIFD'
'DwAAaW5wdXRzL2ZvcmVjYXN0XzIwMjRfdjc1X2NsZWFuLmNzdlBLAQIUAxQAAAAIANEqVV1/wAkMGgIAANAFAAAiAAAAAAAAAAAA'
'AACkgRERAABpbnB1dHMvZm9yZWNhc3RfMjAyNF92Nzlfc3ViNDAuY3N2UEsBAhQDFAAAAAgA0SpVXcQcD7oUAgAA0wUAABwAAAAA'
'AAAAAAAAAKSBaxMAAGlucHV0cy9mb3JlY2FzdF8yMDI0X3Y4MS5jc3ZQSwECFAMUAAAACADRKlVdBKXEqBgCAADWBQAAHAAAAAAA'
'AAAAAAAApIG5FQAAaW5wdXRzL2ZvcmVjYXN0XzIwMjRfdjgzLmNzdlBLAQIUAxQAAAAIANEqVV1lOaJwGgIAANMFAAAiAAAAAAAA'
'AAAAAACkgQsYAABpbnB1dHMvdjg0X3ByaW9yX3JlY29uc3RydWN0ZWQuY3N2UEsBAhQDFAAAAAgA0SpVXcP+sVqRAQAA4wQAACIA'
'AAAAAAAAAAAAAKSBZRoAAGlucHV0cy9mb3JlY2FzdF8yMDI0X3Y1OV90NC40Ny5jc3ZQSwECFAMUAAAACADRKlVd1ntjLYoBAADk'
'BAAAKQAAAAAAAAAAAAAApIE2HAAAaW5wdXRzL2ZvcmVjYXN0XzIwMjRfdjU4X2hhbGZfc3RyZWFtcy5jc3ZQSwECFAMUAAAACADR'
'KlVd6+03jpcBAADgBAAALgAAAAAAAAAAAAAApIEHHgAAaW5wdXRzL2ZvcmVjYXN0XzIwMjRfdjU3X3N0cmVhbV9jYWxpYnJhdGVk'
'LmNzdlBLAQIUAxQAAAAIANEqVV1LfcPIJgEAABoEAAA0AAAAAAAAAAAAAACkgeofAABpbnB1dHMvZm9yZWNhc3RfMjAyNF92NTZf'
'bXVsdGlzdHJlYW1fc3Vic3BhY2VfZWIuY3N2UEsBAhQDFAAAAAgA0SpVXRe4zZQlAQAAGQQAACkAAAAAAAAAAAAAAKSBYiEAAGlu'
'cHV0cy9mb3JlY2FzdF8yMDI0X3Y1NV9zaW1wbGV4M2RfZWIuY3N2UEsBAhQDFAAAAAgA0SpVXexPWiEiAQAAGQQAAC4AAAAAAAAA'
'AAAAAKSBziIAAGlucHV0cy9mb3JlY2FzdF8yMDI0X3Y1NF9zb2Z0X3RocmVzaG9sZF9lYi5jc3ZQSwECFAMUAAAACADRKlVd+LE1'
'2icBAAAZBAAAMAAAAAAAAAAAAAAApIE8JAAAaW5wdXRzL2ZvcmVjYXN0XzIwMjRfdjUzX2h1YmVyX2ViX2F1dG9ub21vdXMuY3N2'
'UEsBAhQDFAAAAAgA0SpVXe/0SxwiAQAAGQQAACwAAAAAAAAAAAAAAKSBsSUAAGlucHV0cy9mb3JlY2FzdF8yMDI0X3Y1Ml9hdXRv'
'bm9tb3VzX21ldGEuY3N2UEsBAhQDFAAAAAgA0SpVXaqnmQ0nAQAAGQQAACoAAAAAAAAAAAAAAKSBHScAAGlucHV0cy9mb3JlY2Fz'
'dF8yMDI0X3Y1MF9ncmFuZF91bmlmaWVkLmNzdlBLAQIUAxQAAAAIANEqVV2yeIPpIQEAABkEAAAtAAAAAAAAAAAAAACkgYwoAABp'
'bnB1dHMvZm9yZWNhc3RfMjAyNF92NDhfcm9idXN0X2NvbnNlbnN1cy5jc3ZQSwECFAMUAAAACADRKlVd/YUpISIBAAAZBAAAKQAA'
'AAAAAAAAAAAApIH4KQAAaW5wdXRzL2ZvcmVjYXN0XzIwMjRfdjQ3X3N1YjQ3X2VuZ2luZS5jc3ZQSwECFAMUAAAACADRKlVdQ9OP'
'cyYBAAAZBAAAKgAAAAAAAAAAAAAApIFhKwAAaW5wdXRzL2ZvcmVjYXN0XzIwMjRfdjQ1X21pbmltYXhfc3ViNTAuY3N2UEsFBgAA'
'AAAWABYAUQcAAM8sAAAAAA=='
)

_bundle = zipfile.ZipFile(BytesIO(base64.b64decode(BUNDLE_B64)))
DATA_DIR = Path(tempfile.mkdtemp(prefix="v84_standalone_"))
_bundle.extractall(DATA_DIR)
MANIFEST_PATH = DATA_DIR / "manifest.json"
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
print(f"Входные CSV встроены: {1 + len(manifest['basis'])}")
print(f"Папка временных данных: {DATA_DIR}")


In [ ]:
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def read_forecast(entry, expected_dates=None):
    path = DATA_DIR / entry["path"]
    if sha256(path) != entry["sha256"]:
        raise ValueError(f"Не совпала контрольная сумма: {entry['path']}")
    frame = pd.read_csv(path, parse_dates=["date"])
    if list(frame.columns) != ["date", "forecast"]:
        raise ValueError(f"Неверные столбцы: {entry['path']}")
    values = frame["forecast"].to_numpy(dtype=float)
    if frame["date"].isna().any() or frame["date"].duplicated().any():
        raise ValueError(f"Некорректные даты: {entry['path']}")
    if not np.isfinite(values).all() or (values < 0).any():
        raise ValueError(f"Некорректные прогнозы: {entry['path']}")
    if expected_dates is not None and not np.array_equal(frame["date"], expected_dates):
        raise ValueError(f"Набор дат отличается: {entry['path']}")
    return frame

base = read_forecast(manifest["baseline"])
base_log = np.log1p(base["forecast"].to_numpy(dtype=float))
active = base_log > 0
rows, rhs = [], []
for item in manifest["basis"]:
    candidate = read_forecast(item, base["date"])
    delta = np.log1p(candidate["forecast"].to_numpy(dtype=float)) - base_log
    if not np.all(np.abs(delta[~active]) < 1e-4):
        raise ValueError(f"Изменена нулевая маска: {item['path']}")
    score = float(item["score"])
    if not np.isfinite(score) or score < 0:
        raise ValueError(f"Некорректный score для {item['path']}")
    rows.append(delta[active])
    rhs.append(0.5 * (
        np.sum(delta[active] ** 2)
        - manifest["config"]["assumed_evaluated_days"]
        * (score**2 - manifest["config"]["reference_score"]**2)
    ))

D = np.ascontiguousarray(rows, dtype=np.float64)
rhs = np.asarray(rhs, dtype=np.float64)
alpha = float(manifest["config"]["regularization"])
weights = np.linalg.solve(D @ D.T + alpha * np.eye(len(rhs)), rhs)
correction = D.T @ weights


In [ ]:
# Возвращаем поправку в лог-пространстве, применяем закрытые дни и масштабируем сумму.
adjusted = base_log.copy()
adjusted[active] += correction
config = manifest["config"]
closed = (base["date"].dt.dayofweek == 6).to_numpy()
closed |= base["date"].dt.strftime("%Y-%m-%d").isin(config["closed_saturdays"]).to_numpy()
adjusted[closed] = 0.0
adjusted = np.maximum(adjusted, 0.0)

counts = np.expm1(adjusted[~closed])
if not np.isfinite(counts).all() or counts.sum() <= 0:
    raise ValueError("После калибровки получен некорректный прогноз")
forecast_values = np.zeros(len(base), dtype=float)
forecast_values[~closed] = np.round(
    counts * (float(config["target_volume"]) / counts.sum()), 6
)
forecast = pd.DataFrame({
    "date": base["date"].dt.strftime("%Y-%m-%d"),
    "forecast": forecast_values,
})

output_path = Path("forecast_2024_v84_standalone.csv").resolve()
forecast.to_csv(output_path, index=False)
output_hash = sha256(output_path)
expected_hash = manifest["reported_result"]["sha256"]
metadata = {
    "method": manifest["method"],
    "independent_historical_model": False,
    "manifest_sha256": sha256(MANIFEST_PATH),
    "basis_count": len(rows),
    "direction_rank": int(np.linalg.matrix_rank(D)),
    "forecast_sum": float(forecast_values.sum()),
    "zero_days": int((forecast_values == 0).sum()),
    "reference_public_score": manifest["reported_result"]["public_rmsle"],
    "public_score_computed_locally": False,
    "matches_reported_csv": output_hash == expected_hash,
    "output_sha256": output_hash,
}
metadata_path = output_path.with_suffix(".metadata.json")
metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(json.dumps(metadata, ensure_ascii=False, indent=2))
print(f"\nСохранён прогноз: {output_path}")
print(f"Сохранены метаданные: {metadata_path}")


## Что делает расчёт

Поправка строится для `log(1 + прогноз)` по разностям между базовым прогнозом V61 и входными прогнозами. Решается регуляризованная система Грама с `alpha = 0.35`; затем значения переводятся обратно, закрытые даты обнуляются, а сумма открытых дат приводится к `11 400`.

`matches_reported_csv` показывает, совпали ли байты результата с сохранённым CSV V84. Совпадение подтверждает воспроизводимость файла, но не проверяет Public score на скрытых данных.
